In [6]:
import numpy as np 
import pandas as pd
import os, sys, glob, tables
import matplotlib.pyplot as plt

from traitlets.config import Config
from ctapipe.visualization import CameraDisplay
from ctapipe.io import EventSeeker, EventSource
from ctapipe.instrument import CameraGeometry
import astropy.units as u

# Loading the camera geometry
config = Config({'LSTEventSource':{'default_trigger_type': 'ucts','allowed_tels': [1],'min_flatfield_adc': 3000, 'min_flatfield_pixel_fraction': 0.8,},})
source = EventSource(input_url="/fefs/aswg/data/real/R0/20241125/LST-1.1.Run19799.0000.fits.fz", config=config, max_events=1)
camgeom = source.subarray.tel[1].camera.geometry

In [2]:
# Non corrected
f0 = "/fefs/aswg/data/real/DL1/20241125/v0.10/tailcut84/dl1_LST-1.Run19799.0000.h5"
# Corrected
f1 = "/fefs/aswg/workspace/juan.jimenez/data/real/mono/S241125n/v0.10.18/GammaDiffuse/prod_light_scaling/DL1/Run19799/dl1_LST-1.Run19799.0000.h5"

table0 = tables.open_file(f0)
table1 = tables.open_file(f1)

In [3]:
ID = 3654 # 22924
ind0 = np.where(table0.root.dl1.event.telescope.image.LST_LSTCam.col('event_id')==ID)[0][0]
ind1 = np.where(table1.root.dl1.event.telescope.image.LST_LSTCam.col('event_id')==ID)[0][0]

In [ ]:
i0 = table0.root.dl1.event.telescope.parameters.LST_LSTCam.col("intensity")[ind0]
i1 = table1.root.dl1.event.telescope.parameters.LST_LSTCam.col("intensity")[ind1]

p0 = table0.root.dl1.event.telescope.parameters.LST_LSTCam.col("n_pixels")[ind0]
p1 = table1.root.dl1.event.telescope.parameters.LST_LSTCam.col("n_pixels")[ind1]

print(f"N-pixels: {p0} - {p1}")
print(f"Intensities: {i0:.2f} - {i1:.2f}")

In [ ]:
d0 = table0.root.dl1.event.telescope.image.LST_LSTCam.col('image')[ind0]
d1 = table1.root.dl1.event.telescope.image.LST_LSTCam.col('image')[ind1]

m0 = table0.root.dl1.event.telescope.image.LST_LSTCam.col('image_mask')[ind0]
m1 = table1.root.dl1.event.telescope.image.LST_LSTCam.col('image_mask')[ind1]

print(sum(m0), sum(m1))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

disp0 = CameraDisplay(camgeom, ax=ax1, show_frame=False, title="Non-Corrected")
disp1 = CameraDisplay(camgeom, ax=ax2, show_frame=False, title="Corrected")
disp0.image, disp1.image = d0, d1
disp0.add_colorbar(label="Charg", ax=ax1); disp1.add_colorbar(label="Charge", ax=ax2)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

disp1 = CameraDisplay(camgeom, ax=ax, show_frame=False, cmap="gnuplot")
disp1.image = np.array(m1).astype(float) + np.array(m0).astype(float) * 1
disp1.add_colorbar(label='Mask claning',ax=ax)

plt.show()